# Lesson 20 Lab — Interpreter, Assertions, and Debugging Tools

**Puzzle:** When compile-time checks, device checks, interpreter limits, and sanitizers change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates compile-time checks, device checks, interpreter limits, and sanitizers and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Triton offers static_print/static_assert for compile time and device_print/device_assert for runtime. TRITON_INTERPRET executes programs sequentially on CPU for inspection, with documented dtype and indirect-address limitations. GPU performance must be measured outside interpreter mode.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["compile-time checks, device checks, interpreter limits, and sanitizers"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: documented installed capability. Candidate: reviewed Triton kernel or explicit model described below.

Debug output can perturb timing, while interpreter success does not prove GPU scheduling or memory behavior.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 20
LESSON_TITLE = 'Interpreter, Assertions, and Debugging Tools'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260833
}


## 5. Freeze the experiment

**Experiment:** Probe the four documented debug operators and retain a separate normal GPU correctness run.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 4,
  "secondary": false,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "debug_operations": {
      "static_print": true,
      "static_assert": true,
      "device_print": true,
      "device_assert": true
    },
    "interpreter_enabled_for_this_gpu_run": false,
    "note": "Interpreter is a separate CPU debugging mode, not a GPU performance mode"
  }
}
All 4 documented debug operators were present. The retained GPU run validates the kernel; interpreter mode remains a separate CPU debugging workflow.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Debug operators present | 4 |
| Interpreter active | false |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

All 4 documented debug operators were present. The retained GPU run validates the kernel; interpreter mode remains a separate CPU debugging workflow.

The installed toolchain or API surface was inspected. An available symbol or source file is not reported as native performance on an unexecuted backend.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use interpreter and assertions to localize errors, then rerun correctness and performance on the target GPU.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 20,
  "title": "Interpreter, Assertions, and Debugging Tools",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260833
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "primary": 4,
    "secondary": false,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "debug_operations": {
        "static_print": true,
        "static_assert": true,
        "device_print": true,
        "device_assert": true
      },
      "interpreter_enabled_for_this_gpu_run": false,
      "note": "Interpreter is a separate CPU debugging mode, not a GPU performance mode"
    }
  },
  "analysis_en": "All 4 documented debug operators were present. The retained GPU run validates the kernel; interpreter mode remains a separate CPU debugg

## 10. Make the bounded decision

> Use interpreter and assertions to localize errors, then rerun correctness and performance on the target GPU.

**Failure analysis:** Debug output can perturb timing, while interpreter success does not prove GPU scheduling or memory behavior.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
